# Lab 3B · Open-vocabulary toolbox

**Computer Vision for Developers · SDAIA Academy** · Day 3 · take-home · Act 5

**Goal:** Detect anything you can name with Grounding DINO, turn the boxes into exact masks with SAM and measure their area, estimate depth from one photo with Depth Anything V2, then find three prompts that break it.

## How to use this notebook

1. Switch the runtime to a **T4 GPU** first: *Runtime → Change runtime type → T4 GPU*.
2. Run the cells **from top to bottom** with *Shift + Enter*. Do not skip cells: later cells use variables from earlier ones.
3. Cells with a **TODO** comment need **one line** from you. The cell right after each TODO is a check: it prints ✅ when your line is right, or a hint when it is not.
4. Nobody writes a cell from empty. If you are stuck for more than five minutes, open the *solved* version of this notebook and compare.
5. Colab forgets files when the session ends. If you come back later, run everything again from the top.
6. The **Extra tasks** at the end are optional and are not graded.

Expected time: about 40 minutes in class; finish the rest at home.

In [ ]:
# GPU check, with a CPU fallback
import os, torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE == "cuda":
    print("✅ GPU:", torch.cuda.get_device_name(0))
else:
    print("⚠️ No GPU found. Runtime → Change runtime type → T4 GPU, then Runtime → Restart.")
    print("   You can keep going on CPU: the notebook switches to CPU (each model runs in a few seconds per image).")
# FAST mode shrinks data and epochs so everything still finishes on a CPU.
FAST = DEVICE == "cpu" or os.environ.get("CVD_FAST") == "1"
print("DEVICE =", DEVICE, "| FAST =", FAST)

In [ ]:
import numpy as np, matplotlib.pyplot as plt, urllib.request
from PIL import Image
import transformers
from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection, SamModel, SamProcessor, pipeline
print("transformers", transformers.__version__)

try:
    urllib.request.urlretrieve("https://ultralytics.com/images/bus.jpg", "bus.jpg")
    image = Image.open("bus.jpg").convert("RGB")
except Exception as e:
    print("Download failed, using a built-in image:", e)
    from skimage import data
    image = Image.fromarray(data.astronaut())
print("image size (W, H):", image.size)
plt.figure(figsize=(5, 6.5)); plt.imshow(image); plt.axis("off"); plt.show()

## 5 · Open-vocabulary detection: Grounding DINO

Give the model **text**; it returns boxes for whatever the text describes. The prompt format matters: **lowercase, with a full stop after each class** — `"a bus. a person."`

In [ ]:
classes = ["Bus", "Person", "Backpack", "Traffic light"]
text = ""  # TODO: lowercase every class and end each with a full stop, joined by spaces → "bus. person. backpack. traffic light."
print(repr(text))

In [ ]:
# Check
try:
    assert text == "bus. person. backpack. traffic light."
    print('✅ Prompt format is right: lowercase, a full stop after each class.')
except Exception as e:
    print('❌ Not yet. Hint: " ".join(c.lower() + "." for c in classes)')

In [ ]:
GD = "IDEA-Research/grounding-dino-tiny"
gd_processor = AutoProcessor.from_pretrained(GD)
gd_model = AutoModelForZeroShotObjectDetection.from_pretrained(GD).to(DEVICE).eval()
inputs = gd_processor(images=image, text=text, return_tensors="pt").to(DEVICE)
with torch.no_grad():
    outputs = gd_model(**inputs)
det = gd_processor.post_process_grounded_object_detection(
    outputs, inputs.input_ids, threshold=0.35, text_threshold=0.25, target_sizes=[image.size[::-1]])[0]
phrases = det.get("text_labels", det["labels"])          # v5 returns text_labels; older versions put text in labels
boxes = det["boxes"].cpu()
for p, s, b in zip(phrases, det["scores"].tolist(), boxes.tolist()):
    print(f"{p:<15} {s:.2f}  {[round(v) for v in b]}")

In [ ]:
def draw(ax, image, boxes, labels, color="lime"):
    ax.imshow(image); ax.axis("off")
    for (x1, y1, x2, y2), l in zip(boxes, labels):
        ax.add_patch(plt.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, color=color, lw=2))
        ax.text(x1, y1, l, color="black", fontsize=8, bbox=dict(facecolor=color, alpha=0.8, pad=1))
fig, ax = plt.subplots(figsize=(5, 6.5)); draw(ax, image, boxes.tolist(), phrases); plt.show()

## 5.1 · Promptable segmentation: SAM

SAM does not know *what* things are — it returns the exact pixels of whatever you point at. Here we point with the Grounding DINO boxes: **text → box → mask** ("Grounded SAM").

In [ ]:
SAM = "facebook/sam-vit-base"
sam_processor = SamProcessor.from_pretrained(SAM)
sam_model = SamModel.from_pretrained(SAM).to(DEVICE).eval()
sam_inputs = sam_processor(images=image, input_boxes=[boxes.tolist()], return_tensors="pt").to(DEVICE)
with torch.no_grad():
    sam_out = sam_model(**sam_inputs, multimask_output=False)
masks = sam_processor.image_processor.post_process_masks(
    sam_out.pred_masks.cpu(), sam_inputs["original_sizes"].cpu(), sam_inputs["reshaped_input_sizes"].cpu())[0][:, 0]
masks = masks.numpy().astype(bool)
print("masks:", masks.shape, "→ (boxes, H, W)", "| IoU scores:", sam_out.iou_scores.squeeze().cpu().numpy().round(2))

In [ ]:
overlay = np.array(image).astype(float) / 255
rng = np.random.default_rng(1)
for m in masks:
    overlay[m] = overlay[m] * 0.45 + rng.uniform(0.2, 1, 3) * 0.55
plt.figure(figsize=(5, 6.5)); plt.imshow(overlay); plt.axis("off"); plt.title("text → box → mask"); plt.show()

### 5.2 · Measure: mask area
Boxes can count; masks can **measure**. Report every object's area in pixels and as a share of the photo.

In [ ]:
areas = []
for p, m in zip(phrases, masks):
    area_pct = 0.0  # TODO: percentage of the image covered by mask m (pixels in the mask ÷ all pixels × 100)
    areas.append(area_pct)
    print(f"{p:<15} {int(m.sum()):>8} px   {area_pct:5.1f}% of the image")

In [ ]:
# Check
try:
    assert len(areas) == len(masks) and all(abs(a - 100 * m.mean()) < 1e-6 for a, m in zip(areas, masks))
    print('✅ Areas are correct.')
except Exception as e:
    print('❌ Not yet. Hint: area_pct = 100 * m.sum() / m.size')

## 5.3 · Depth from one photo: Depth Anything V2

A relative depth map from a single image, with no training. Depth Anything predicts **relative inverse depth**: larger values = **closer** to the camera.

In [ ]:
depth_pipe = pipeline("depth-estimation", model="depth-anything/Depth-Anything-V2-Small-hf", device=0 if DEVICE == "cuda" else -1)
dres = depth_pipe(image)
depth = dres["predicted_depth"].squeeze().cpu().numpy()
depth = np.array(Image.fromarray(depth).resize(image.size, Image.BILINEAR))
fig, axes = plt.subplots(1, 2, figsize=(10, 6.5))
axes[0].imshow(image); axes[0].axis("off")
axes[1].imshow(dres["depth"], cmap="inferno"); axes[1].axis("off"); axes[1].set_title("bright = near")
plt.show()

In [ ]:
mean_depth = [float(depth[m].mean()) for m in masks]
nearest = 0  # TODO: index of the object with the LARGEST mean depth value (= nearest), np.argmax
for p, d in zip(phrases, mean_depth):
    print(f"{p:<15} mean depth {d:6.2f}")
print("nearest object:", phrases[nearest])

In [ ]:
# Check
try:
    assert nearest == int(np.argmax(mean_depth))
    print('✅ Nearest object found by combining mask and depth.')
except Exception as e:
    print('❌ Not yet. Hint: nearest = int(np.argmax(mean_depth))')

## 5.4 · Break it: three prompts that fail

Foundation models fail in ways you can predict from the slides (counting, spatial words, negation, rare words, things that are not there). Try your own prompts in the cell below, then fill in the table.

In [ ]:
def ground(prompt, thr=0.35):
    inp = gd_processor(images=image, text=prompt, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        o = gd_model(**inp)
    r = gd_processor.post_process_grounded_object_detection(o, inp.input_ids, threshold=thr, text_threshold=0.25,
                                                            target_sizes=[image.size[::-1]])[0]
    labs = r.get("text_labels", r["labels"])
    fig, ax = plt.subplots(figsize=(4, 5.2)); draw(ax, image, r["boxes"].cpu().tolist(), labs, "orange"); ax.set_title(prompt, fontsize=9); plt.show()

for prompt in ["the person on the far left.", "a person not wearing a jacket.", "a giraffe."]:
    ground(prompt)

| Prompt | What happened | Why (which failure type?) |
|---|---|---|
| 1. | | |
| 2. | | |
| 3. | | |

## Extra tasks (optional)

1. **Your own photo:** upload a photo (Files panel), and run the whole text → box → mask → area pipeline on it.
2. **Point prompts:** give SAM a single click (`input_points=[[[x, y]]]`) on the bus instead of a box.
3. **Thresholds:** run Grounding DINO with `threshold=0.2` and `0.5`. What appears and disappears?
4. **CLIP check:** crop each detected box and ask CLIP (Lab 3A) whether the crop matches its phrase — a cheap way to filter false positives.

In [ ]:
# Extra 1 — your code here

In [ ]:
# Extra 2 — your code here (hint: sam_processor(images=image, input_points=[[[x, y]]], ...))

In [ ]:
# Extra 3 — your code here

In [ ]:
# Extra 4 — your code here